In [1]:
! uv add shutil

  × No solution found when resolving dependencies for split
  │ (python_full_version >= '3.14'):
  ╰─▶ Because shutil was not found in the package registry and your project
      depends on shutil, we can conclude that your project's requirements
      are unsatisfiable.

      hint: While the active Python version is 3.11, the resolution failed for
      other Python versions supported by your project. Consider limiting your
      project's supported Python versions using `requires-python`.
  help: If you want to add the package regardless of the failed resolution,
        provide the `--frozen` flag to skip locking and syncing.


In [2]:
import os
import json
import subprocess
import shutil
from pathlib import Path

# ─────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────
FRAMES_DIR      = "changed_frames"
AUDIO_DIR       = "audio_clips"
TEMP_DIR        = "temp_clips"
OUTPUT_VIDEO    = "final_demo.mp4"
AUDIO_META_PATH = os.path.join(AUDIO_DIR, "audio_meta.json")

# Video settings
FPS             = 30
RESOLUTION      = (1920, 1080)   # match your screen recording resolution
TRANSITION_SECS = 0.3            # fade between clips

os.makedirs(TEMP_DIR, exist_ok=True)

# ─────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────
def run(cmd, label=""):
    """Run an ffmpeg command and print result"""
    print(f"\n  ▶ {label}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  ❌ FFmpeg error:\n{result.stderr}")
        raise RuntimeError(f"FFmpeg failed: {label}")
    return result

def get_audio_duration(audio_path):
    """Get duration of an audio file in seconds using ffprobe"""
    result = subprocess.run([
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        audio_path
    ], capture_output=True, text=True)
    return float(result.stdout.strip())

def frame_to_video_clip(frame_path, audio_path, output_path, duration):
    """
    Combine a single frame image + audio into a video clip.
    The frame is held as a still image for the duration of the audio.
    """
    run([
        "ffmpeg", "-y",
        "-loop", "1",                          # loop the still image
        "-i", frame_path,                      # input: image
        "-i", audio_path,                      # input: audio
        "-c:v", "libx264",                     # video codec
        "-tune", "stillimage",                 # optimized for still frames
        "-c:a", "aac",                         # audio codec
        "-b:a", "192k",                        # audio bitrate
        "-pix_fmt", "yuv420p",                 # pixel format (compatibility)
        "-vf", f"scale={RESOLUTION[0]}:{RESOLUTION[1]}:force_original_aspect_ratio=decrease,"
               f"pad={RESOLUTION[0]}:{RESOLUTION[1]}:(ow-iw)/2:(oh-ih)/2",
        "-shortest",                           # end when audio ends
        "-t", str(duration + 0.1),             # safety buffer
        output_path
    ], label=f"Frame → Clip: {Path(frame_path).name}")

def add_fade(input_path, output_path, duration, fade_duration=0.3):
    """Add fade-in and fade-out to a clip"""
    fade_out_start = max(0, duration - fade_duration)
    run([
        "ffmpeg", "-y",
        "-i", input_path,
        "-vf",
        f"fade=t=in:st=0:d={fade_duration},"
        f"fade=t=out:st={fade_out_start}:d={fade_duration}",
        "-af",
        f"afade=t=in:st=0:d={fade_duration},"
        f"afade=t=out:st={fade_out_start}:d={fade_duration}",
        output_path
    ], label=f"Adding fade: {Path(input_path).name}")

def concatenate_clips(clip_paths, output_path):
    """Concatenate all clips into one final video using concat demuxer"""
    concat_list = os.path.join(TEMP_DIR, "concat_list.txt")

    with open(concat_list, "w") as f:
        for clip in clip_paths:
            f.write(f"file '{os.path.abspath(clip)}'\n")

    run([
        "ffmpeg", "-y",
        "-f", "concat",
        "-safe", "0",
        "-i", concat_list,
        "-c:v", "libx264",
        "-c:a", "aac",
        "-b:a", "192k",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",   # web-optimized (loads faster)
        output_path
    ], label="Concatenating all clips → final video")

# ─────────────────────────────────────────
# STEP 1: LOAD AUDIO METADATA
# ─────────────────────────────────────────
with open(AUDIO_META_PATH) as f:
    audio_meta = json.load(f)

print(f"🎬 Total clips to assemble: {len(audio_meta)}")

# ─────────────────────────────────────────
# STEP 2: BUILD INDIVIDUAL CLIPS
# ─────────────────────────────────────────
clip_paths = []

for i, entry in enumerate(audio_meta):
    frame_name = entry["frame"]
    audio_name = entry["audio"]
    position   = entry.get("position", "middle")

    frame_path = os.path.join(FRAMES_DIR, frame_name)
    audio_path = os.path.join(AUDIO_DIR,  audio_name)

    # Output paths
    raw_clip    = os.path.join(TEMP_DIR, f"clip_{i:04d}_raw.mp4")
    faded_clip  = os.path.join(TEMP_DIR, f"clip_{i:04d}_faded.mp4")

    print(f"\n[{i+1}/{len(audio_meta)}] {frame_name} ({position})")

    # Check files exist
    if not os.path.exists(frame_path):
        print(f"  ⚠️  Frame not found: {frame_path}, skipping.")
        continue
    if not os.path.exists(audio_path):
        print(f"  ⚠️  Audio not found: {audio_path}, skipping.")
        continue

    # Get audio duration — clip holds the frame for this long
    duration = get_audio_duration(audio_path)
    print(f"  ⏱️  Duration: {round(duration, 2)}s")

    # Step A: Combine frame + audio into clip
    frame_to_video_clip(frame_path, audio_path, raw_clip, duration)

    # Step B: Add fade in/out (skip fade on opening/closing for cleaner look)
    if position in ("opening", "closing"):
        faded_clip = raw_clip   # no fade on first/last
    else:
        add_fade(raw_clip, faded_clip, duration, TRANSITION_SECS)

    clip_paths.append(faded_clip)
    print(f"  ✅ Clip ready: clip_{i:04d}_faded.mp4")

# ─────────────────────────────────────────
# STEP 3: CONCATENATE INTO FINAL VIDEO
# ─────────────────────────────────────────
print(f"\n{'─'*55}")
print(f"🔗 Concatenating {len(clip_paths)} clips...")

concatenate_clips(clip_paths, OUTPUT_VIDEO)

# ─────────────────────────────────────────
# STEP 4: CLEANUP TEMP FILES
# ─────────────────────────────────────────
shutil.rmtree(TEMP_DIR)
print(f"🧹 Cleaned up temp files")

# ─────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────
video_size = os.path.getsize(OUTPUT_VIDEO) / (1024 * 1024)

print(f"\n{'─'*55}")
print(f"✅ FINAL VIDEO READY")
print(f"{'─'*55}")
print(f"Output:     {OUTPUT_VIDEO}")
print(f"Clips used: {len(clip_paths)}")
print(f"Size:       {round(video_size, 2)} MB")
print(f"{'─'*55}")

🎬 Total clips to assemble: 17

[1/17] frame_01.png (opening)
  ⚠️  Frame not found: changed_frames\frame_01.png, skipping.

[2/17] frame_02.png (early)
  ⚠️  Frame not found: changed_frames\frame_02.png, skipping.

[3/17] frame_03.png (early)
  ⚠️  Frame not found: changed_frames\frame_03.png, skipping.

[4/17] frame_04.png (early)
  ⚠️  Frame not found: changed_frames\frame_04.png, skipping.

[5/17] frame_05.png (early)
  ⚠️  Frame not found: changed_frames\frame_05.png, skipping.

[6/17] frame_06.png (early)
  ⚠️  Frame not found: changed_frames\frame_06.png, skipping.

[7/17] frame_07.png (middle)
  ⚠️  Frame not found: changed_frames\frame_07.png, skipping.

[8/17] frame_08.png (middle)
  ⚠️  Frame not found: changed_frames\frame_08.png, skipping.

[9/17] frame_09.png (middle)
  ⚠️  Frame not found: changed_frames\frame_09.png, skipping.

[10/17] frame_10.png (middle)
  ⚠️  Frame not found: changed_frames\frame_10.png, skipping.

[11/17] frame_11.png (middle)
  ⚠️  Frame not found:

RuntimeError: FFmpeg failed: Concatenating all clips → final video

🎬 Total frames: 89
🎙️  Audio clips:  17

🔧 Calculating timestamps from audio durations...
  [01]    0.00s →   16.95s | Welcome to VisionCurator, the platform designed to help...
  [02]   16.95s →   27.07s | On the Datasets page, you can see an overview of your u...
  [03]   27.07s →   36.64s | Let's explore our 'dogs' dataset. VisionCurator automat...
  [04]   36.64s →   43.51s | You can easily browse through all images, gaining immed...
  [05]   43.51s →   51.59s | VisionCurator intelligently organizes your images into ...
  [06]   51.59s →   60.84s | Here, we see images grouped into distinct clusters, lik...
  [07]   60.84s →   70.91s | This clustering capability is crucial for identifying u...
  [08]   70.91s →   80.20s | Beyond clusters, VisionCurator also identifies outliers...
  [09]   80.20s →   89.44s | These outliers represent unique or unusual examples tha...
  [10]   89.44s →   97.38s | Now, let's leverage this intelligence with VisionCurato...
  [11]   97.38s →  108.53s | S

In [1]:
import os
import json
import subprocess
from pathlib import Path

# ─────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────
ORIGINAL_VIDEO = "video.mp4"
AUDIO_DIR = "audio_clips"
OUTPUT_VIDEO = "final_demo.mp4"
SUBTITLES_FILE = "subtitles.srt"

AUDIO_META_PATH = os.path.join(AUDIO_DIR, "audio_meta.json")

# ─────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────
def run(cmd, label=""):
    print(f"▶ {label}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(label)
    return result


def get_duration(path):
    r = subprocess.run([
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        path
    ], capture_output=True, text=True)
    return float(r.stdout.strip())


def seconds_to_srt(s):
    ms = int((s % 1) * 1000)
    ss = int(s) % 60
    m = int(s // 60) % 60
    h = int(s // 3600)
    return f"{h:02d}:{m:02d}:{ss:02d},{ms:03d}"


# ─────────────────────────────────────────
# STEP 1: LOAD META
# ─────────────────────────────────────────
with open(AUDIO_META_PATH) as f:
    audio_meta = json.load(f)

video_duration = get_duration(ORIGINAL_VIDEO)
print("Video duration:", video_duration)

audio_entries = []

# ─────────────────────────────────────────
# STEP 2: PREPARE AUDIO TIMELINE
# ─────────────────────────────────────────
for entry in audio_meta:

    audio_path = os.path.join(AUDIO_DIR, entry["audio"])

    if not os.path.exists(audio_path):
        continue

    dur = get_duration(audio_path)

    # REAL timestamp from frame detection
    start_s = float(entry["timestamp"])

    # small anticipation offset (optional)
    start_s = max(0, start_s - 0.4)

    end_s = start_s + dur

    audio_entries.append({
        "audio_path": audio_path,
        "start": start_s,
        "end": end_s,
        "duration": dur,
        "text": entry.get("narration", "")
    })

# sort by timestamp
audio_entries.sort(key=lambda x: x["start"])

# ─────────────────────────────────────────
# STEP 3: BUILD AUDIO TRACK
# ─────────────────────────────────────────
print("Building narration track...")

inputs = ["-f", "lavfi", "-i", "anullsrc=r=44100:cl=stereo"]

filter_parts = []
streams = []

for i, ae in enumerate(audio_entries):

    delay = int(ae["start"] * 1000)

    inputs += ["-i", ae["audio_path"]]

    filter_parts.append(
        f"[{i+1}:a]adelay={delay}|{delay}[a{i}]"
    )

    streams.append(f"[a{i}]")

mix_inputs = "".join(streams)

filter_parts.append(
    f"{mix_inputs}amix=inputs={len(streams)}[mix]"
)

filter_complex = ";".join(filter_parts)

AUDIO_OUTPUT = "narration_track.aac"

run(
    ["ffmpeg", "-y"] + inputs + [
        "-filter_complex", filter_complex,
        "-map", "[mix]",
        "-c:a", "aac",
        "-b:a", "192k",
        AUDIO_OUTPUT
    ],
    "Creating narration track"
)

# ─────────────────────────────────────────
# STEP 4: GENERATE SUBTITLES
# ─────────────────────────────────────────
print("Generating subtitles...")

srt_blocks = []

for i, ae in enumerate(audio_entries):

    srt_blocks.append(
        f"{i+1}\n"
        f"{seconds_to_srt(ae['start'])} --> {seconds_to_srt(ae['end'])}\n"
        f"{ae['text']}"
    )

with open(SUBTITLES_FILE, "w", encoding="utf-8") as f:
    f.write("\n\n".join(srt_blocks))

# ─────────────────────────────────────────
# STEP 5: FINAL MERGE
# ─────────────────────────────────────────
print("Rendering final video...")

run([
    "ffmpeg", "-y",
    "-i", ORIGINAL_VIDEO,
    "-i", AUDIO_OUTPUT,
    "-map", "0:v",
    "-map", "1:a",
    "-vf", f"subtitles={SUBTITLES_FILE}",
    "-c:v", "libx264",
    "-preset", "fast",
    "-crf", "18",
    "-c:a", "aac",
    "-b:a", "192k",
    "-shortest",
    OUTPUT_VIDEO
], "Final render")

print("Final video ready:", OUTPUT_VIDEO)

Video duration: 88.8584


KeyError: 'timestamp'